# 11. 일반화 실험문제와 가설 정의

이 노트북은 2장의 실험 문제를 실제 데이터셋 metadata 기준으로 고정합니다.

목표는 가정으로 결론을 쓰는 것이 아니라, 이후 노트북에서 SegFormer 추론 결과로 검증할 수 있는 귀무가설과 평가 단위를 명확히 만드는 것입니다.

In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "ch2_utils.py").exists():
    matches = (
        list(Path.cwd().glob("Deeplearning/*/2-1장/ch2_utils.py"))
        + list(Path.cwd().glob("Deeplearning/*/2장/ch2_utils.py"))
        + list(Path.cwd().glob("**/ch2_utils.py"))
    )
    NOTEBOOK_DIR = matches[0].parent if matches else Path("Deeplearning") / "Vision 응용" / "2-1장"
sys.path.append(str(NOTEBOOK_DIR))

from ch2_utils import *

paths = find_paths()
set_korean_font()
set_seed(7)
samples = load_samples(paths.data_root)
paths

## 11-1. 실제 데이터셋 metadata 확인

In [ ]:
print(f"num_samples = {len(samples)}")
display(samples[["sample_id", "split", "domain_id", "color_group", "shape_group", "defect_type", "image_path", "mask_path"]].head())

group_summary = summarize_groups(samples)
group_summary_path = paths.runs_root / "group_summary.csv"
group_summary.to_csv(group_summary_path, index=False, encoding="utf-8-sig")
display(group_summary.head(20))
print(group_summary_path)

## 11-2. 색상, 형상, 불량 유형 분포

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
samples["color_group"].value_counts().sort_index().plot(kind="bar", ax=axes[0], color="#2563eb", title="color_group")
samples["shape_group"].value_counts().sort_index().plot(kind="bar", ax=axes[1], color="#16a34a", title="shape_group")
samples["defect_type"].value_counts().sort_index().plot(kind="bar", ax=axes[2], color="#dc2626", title="defect_type")
for ax in axes:
    ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
fig.savefig(paths.runs_root / "11_dataset_factor_counts.png", dpi=150)
plt.show()

## 11-3. 귀무가설을 파일로 저장

In [ ]:
hypotheses = [
    {
        "id": "H0-1",
        "name": "색상 노출 효과 없음",
        "null": "학습 데이터의 특정 색상 노출 비율은 같은 defect type의 hold-out 성능에 영향을 주지 않는다.",
        "evidence": "exposure_ratio별 target_dice, target_fnr 변화와 bootstrap CI",
    },
    {
        "id": "H0-2",
        "name": "불량 유형 전이 없음",
        "null": "빨간 scratch 데이터를 많이 추가해도 impact, dent, stain 성능은 변하지 않는다.",
        "evidence": "red_scratch exposure 증가 전후의 다른 defect type 성능 변화",
    },
    {
        "id": "H0-3",
        "name": "형상 변화 효과 없음",
        "null": "metal shape 변화는 같은 색상과 같은 defect type 조건의 성능에 영향을 주지 않는다.",
        "evidence": "shape_group별 target_dice, target_fnr 비교",
    },
    {
        "id": "H0-4",
        "name": "상호작용 효과 없음",
        "null": "color, shape, defect type 사이에는 상호작용 효과가 없다.",
        "evidence": "color x shape x defect heatmap과 worst group gap",
    },
]
experiment_plan = {
    "dataset_root": str(paths.data_root),
    "model": "nvidia/segformer-b0-finetuned-ade-512-512",
    "primary_metric": "target_dice",
    "secondary_metrics": ["target_iou", "target_fnr", "target_fpr", "metal_iou"],
    "hypotheses": hypotheses,
}
save_json(paths.runs_root / "experiment_plan.json", experiment_plan)
display(pd.DataFrame(hypotheses))

## 11-4. 이 노트북의 결론

In [ ]:
color_n = samples["color_group"].nunique()
shape_n = samples["shape_group"].nunique()
defect_n = samples["defect_type"].nunique()
print(
    f"결론: 현재 데이터셋은 color {color_n}개, shape {shape_n}개, defect {defect_n}개 그룹을 포함합니다. "
    "따라서 2장에서는 전체 평균이 아니라 color/shape/defect factor별 SegFormer 추론 성능을 분리해서 검증합니다."
)